 ## Notebook 2.1

To improve our model and get a higher accuracy than in notebook 2, we are now trying with the use of Silu to get highier percentages 
- **SiLU Architecture (`efficientnet_v2_s`):** Uses SiLU/Swish activation functions, which help prevent gradient vanishing in fine strokes and paleographic details.
- **High Resolution (`384 × 384`):** Almost triples the number of pixels compared to `224 × 224`, allowing the model to capture the ductus, ligatures, and the exact style of medieval pen strokes.
- **Weighted Loss (`Weighted CrossEntropyLoss`):** higher penalty to difficult classes (Types 6, 8, and 9), forcing the model not to ignore them.
- **Cosine Scheduler (`CosineAnnealingLR`) + Fine-Tuning for 20 Epochs:** Keeps the Learning Rate low and controlled to avoid destroying the pretrained knowledge.
- **Manuscript-Specific Data Augmentation:** Uses `RandomResizedCrop`, `ColorJitter` (to simulate variations in ink brightness/contrast), and `RandomRotation`.

In [ ]:
import os, glob, time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image, ImageFile
import matplotlib.pyplot as plt

# img grandes 
Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True


# Dispositivo y Parametros
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Dispositivo activo: {device}")

BATCH_SIZE = 16          # Reducido por la mayor resolución (384x384)
IMG_SIZE = 384           # Alta resolucion para detalles 
EPOCHS = 20              # 20 epocas con Cosine Annealing
LEARNING_RATE = 1e-4     # LR bajo para ajuste fino preciso

# Metadata y Mapa de Imagenes
DATA_DIR = "/kaggle/input/datasets/mariajgamboa/clammdst/CLaMM_DST"
CSV_PATH = os.path.join(DATA_DIR, "CLaMM_Metadata.csv")

df = pd.read_csv(CSV_PATH, sep=None, engine='python')
df.columns = df.columns.str.strip()

script_col = [col for col in df.columns if col.upper() == 'SCRIPT_TYPE'][0]
df['label'] = df[script_col].astype(int) - 1
num_classes = df['label'].nunique()

all_image_paths = (
    glob.glob(f"{DATA_DIR}/**/*.[jJ][pP][gG]", recursive=True) +
    glob.glob(f"{DATA_DIR}/**/*.[pP][nN][gG]", recursive=True) +
    glob.glob(f"{DATA_DIR}/**/*.[tT][iI][fF]", recursive=True) +
    glob.glob(f"{DATA_DIR}/**/*.[jJ][pP][eE][gG]", recursive=True)
)
image_map = {os.path.basename(p).lower(): p for p in all_image_paths}

# Dataset y Transformaciones de Alta Resolución + SiLU
class CLaMMDataset(Dataset):
    def __init__(self, df, image_map, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_map = image_map
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = str(row['FILENAME']).strip().lower()

        if img_name in self.image_map:
            final_path = self.image_map[img_name]
        else:
            base_name = os.path.splitext(img_name)[0]
            matched = None
            for ext in ['.jpg', '.jpeg', '.tif', '.png']:
                if (base_name + ext) in self.image_map:
                    matched = self.image_map[base_name + ext]
                    break
            final_path = matched

        image = Image.open(final_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(row['label'], dtype=torch.long)

# Augmentation avanzado para manuscritos a 384x384
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomRotation(degrees=8),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 80% train, 20% val
full_dataset = CLaMMDataset(df=df, image_map=image_map, transform=train_tf)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset_split = random_split(
    full_dataset, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

# transformaciones de validacion puras sin augmentation
val_dataset = CLaMMDataset(df=df, image_map=image_map, transform=val_tf)
val_dataset = torch.utils.data.Subset(val_dataset, val_dataset_split.indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


#  Pesos para Clases Desbalanceadas

class_counts = df['label'].value_counts().sort_index().values
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum() * num_classes
class_weights = class_weights.to(device)

print(" Pesos asignados a las clases para corregir el desbalance:")
for i, w in enumerate(class_weights):
    print(f"  └─ Tipo {i+1}: {w.item():.2f}")


#  EfficientNet-V2-S (Usa SiLU Nativamente)

print("\n Cargando EfficientNet-V2-S (con activaciones SiLU)...")
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

# Reemplazar la capa clasificadora final
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, num_classes)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)


# Entrenamiento de Alto Rendimiento

best_val_acc = 0.0
save_filename = "efficientnet_v2_silu_384_clamm.pth"

print(f"\n Iniciando entrenamiento a {EPOCHS} épocas en resolución {IMG_SIZE}x{IMG_SIZE}...\n")
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels.data)
        total_train += labels.size(0)

    scheduler.step()

    train_loss = running_loss / total_train
    train_acc = correct_train.double() / total_train

    # Fase de Validación
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels.data)
            total_val += labels.size(0)

    val_loss = val_loss / total_val
    val_acc = correct_val.double() / total_val

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Época [{epoch+1:02d}/{EPOCHS:02d}] | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # Guardar el mejor modelo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_filename)
        print(f"¡Nuevo récord de precisión en validación: {best_val_acc:.4f}! Modelo guardado.")

elapsed = time.time() - start_time
print(f"\n Entrenamiento completado en {elapsed/60:.2f} minutos.")
print(f" Mejor Accuracy en Validación: {best_val_acc * 100:.2f}%")
print(f" Archivo guardado como: '{save_filename}'")

 Dispositivo activo: cuda
 Pesos asignados a las clases para corregir el desbalance:
  └─ Tipo 1: 0.62
  └─ Tipo 2: 0.59
  └─ Tipo 3: 1.38
  └─ Tipo 4: 1.47
  └─ Tipo 5: 1.33
  └─ Tipo 6: 0.79
  └─ Tipo 7: 0.58
  └─ Tipo 8: 1.07
  └─ Tipo 9: 0.91
  └─ Tipo 10: 1.45
  └─ Tipo 11: 0.40
  └─ Tipo 12: 1.41

 Cargando EfficientNet-V2-S (con activaciones SiLU)...
Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 184MB/s]



 Iniciando entrenamiento a 20 épocas en resolución 384x384...

Época [01/20] | LR: 0.000099 | Train Loss: 1.6667 | Train Acc: 0.3833 | Val Loss: 1.1150 | Val Acc: 0.5573
¡Nuevo récord de precisión en validación: 0.5573! Modelo guardado.
Época [02/20] | LR: 0.000098 | Train Loss: 1.0740 | Train Acc: 0.5636 | Val Loss: 0.9636 | Val Acc: 0.6046
¡Nuevo récord de precisión en validación: 0.6046! Modelo guardado.
Época [03/20] | LR: 0.000095 | Train Loss: 0.8761 | Train Acc: 0.6495 | Val Loss: 0.8068 | Val Acc: 0.6751
¡Nuevo récord de precisión en validación: 0.6751! Modelo guardado.
Época [04/20] | LR: 0.000091 | Train Loss: 0.7434 | Train Acc: 0.6999 | Val Loss: 0.8126 | Val Acc: 0.6861
¡Nuevo récord de precisión en validación: 0.6861! Modelo guardado.
Época [05/20] | LR: 0.000086 | Train Loss: 0.6227 | Train Acc: 0.7478 | Val Loss: 0.7617 | Val Acc: 0.7192
¡Nuevo récord de precisión en validación: 0.7192! Modelo guardado.
Época [06/20] | LR: 0.000080 | Train Loss: 0.5480 | Train Acc: 0.7